In [12]:
import os

DATA_DIR = "/kaggle/input/competitions/birdclef-2026"
OUT_DIR  = "/kaggle/working"
os.makedirs(OUT_DIR, exist_ok=True)

os.listdir(DATA_DIR)


: 

# BirdCLEF+ 2026 — Acoustic Species Identification
**Goal:** Identify 234 wildlife species from 5-second audio windows using mel spectrograms.  
**Stack:** PyTorch (data loading) · dnp (modeling & autograd) · librosa (audio)

In [13]:
## 1 · Install dependencies
import subprocess, sys, os

def pip_install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

pip_install("librosa")
pip_install("soundfile")

print("Dependencies ready.")


Dependencies ready.


In [14]:

## 0 · Environment Setup
import subprocess, sys, os
from pathlib import Path

# ── GPU info ─────────────────────────────────────────────────────────────
print("── GPU info ────────────────────────────────────────────────────")
try:
    r = subprocess.run(['nvidia-smi','--query-gpu=name,memory.total,driver_version',
                        '--format=csv,noheader'], capture_output=True, text=True, timeout=10)
    for line in r.stdout.strip().split('\n'):
        print(f"  {line}")
except Exception as e:
    print(f"  {e}")

# ── Clone or pull repo (branch v3) ──────────────────────────────────────
REPO_DIR = Path('/content/AutoDiff-Numpy')
if not REPO_DIR.exists():
    print("\n── Cloning Seydifa/AutoDiff-Numpy (branch v3) ──────────────────")
    subprocess.run(
        ['git', 'clone', '--branch', 'v3', '--depth', '1',
         'https://github.com/Seydifa/AutoDiff-Numpy.git', str(REPO_DIR)],
        check=True,
    )
    print("  Cloned OK")
else:
    print(f"\n── Pulling latest changes in {REPO_DIR} ─────────────────────")
    result = subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'],
                            capture_output=True, text=True)
    print(f"  {result.stdout.strip() or result.stderr.strip()}")

if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

# ── Numpy: ensure ≥ 2.0 so CuPy 14 and RAPIDS packages work ─────────────
import numpy as np
current_np = tuple(int(x) for x in np.__version__.split(".")[:2])
if current_np < (2, 0):
    print(f"\n── numpy {np.__version__} < 2.0  — upgrading ────────────────────────")
    r_np = subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                           "numpy>=2.0", "--upgrade"],
                          capture_output=True, text=True)
    print("  numpy upgraded OK" if r_np.returncode == 0 else f"  upgrade failed: {r_np.stderr[:200]}")
    print("\n⚠️  KERNEL RESTART REQUIRED to pick up new numpy.")
    print("   → Kernel → Restart Kernel, then re-run from cell 1.")
    raise SystemExit("Restart the kernel now and re-run all cells.")
else:
    print(f"\n  numpy {np.__version__} ✅")

# ── Install remaining deps ────────────────────────────────────────────────
print("\n── Installing remaining dependencies ────────────────────────────")
for pkg in ["librosa", "soundfile", "scikit-learn", "tqdm"]:
    r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg],
                       capture_output=True, text=True)
    status = "OK" if r.returncode == 0 else f"FAILED: {r.stderr[:80]}"
    print(f"  {pkg:15s} {status}")

# ── Verify CuPy ───────────────────────────────────────────────────────────
print("\n── Checking CuPy ───────────────────────────────────────────────")
try:
    import importlib
    importlib.invalidate_caches()
    import cupy as cp
    print(f"  CuPy {cp.__version__} ✅  CUDA {cp.cuda.runtime.runtimeGetVersion()}")
except Exception as e:
    print(f"  CuPy not available: {e}")


── GPU info ────────────────────────────────────────────────────
  Tesla P100-PCIE-16GB, 16384 MiB, 580.105.08

── Pulling latest changes in /content/AutoDiff-Numpy ─────────────────────
  Already up to date.

  numpy 2.0.2 ✅

── Installing remaining dependencies ────────────────────────────
  librosa         OK
  soundfile       OK
  scikit-learn    OK
  tqdm            OK

── Checking CuPy ───────────────────────────────────────────────
  CuPy 14.0.1 ✅  CUDA 12090


In [15]:
## 2 · Imports
import os, math, warnings, importlib
import numpy as np
import pandas as pd
import librosa
import soundfile as sf
from pathlib import Path

import torch
from torch.utils.data import Dataset, DataLoader

import dnp
import dnp.core.backend as _backend

# Invalidate import caches so freshly-installed CuPy is visible, then reload
importlib.invalidate_caches()
importlib.reload(_backend)

# If CuPy is now importable but backend didn't pick it up, patch manually
if not _backend.is_cuda_available:
    try:
        import cupy as _cp
        _backend.is_cuda_available = True
        _backend.backend = _cp
        print("CuPy patched into dnp backend manually.")
    except ImportError:
        print("CuPy not importable — running on CPU.")

from dnp.core.layers     import (Module, Sequential, Conv2d, BatchNorm2d,
                                  Linear, ReLU, GELU, Dropout, Flatten,
                                  BCEWithLogitsLoss)
from dnp.core.optimizers import Adam, ReduceLROnPlateau
from dnp.core.session    import session
from dnp.core.tensor     import Tensor
from dnp.core.backend    import (is_cuda_available, as_cupy, as_numpy,
                                  set_dtype, synchronize)

warnings.filterwarnings("ignore")
np.random.seed(42)

set_dtype("float32")

DEVICE = "cuda" if is_cuda_available else "cpu"
print(f"dnp version : {dnp.__version__ if hasattr(dnp, '__version__') else 'local'}")
print(f"numpy       : {np.__version__}")
print(f"torch       : {torch.__version__}")
print(f"CUDA (CuPy) : {is_cuda_available}  →  device = {DEVICE}")

if is_cuda_available:
    import cupy as cp
    print(f"CuPy        : {cp.__version__}")
    print(f"GPU         : {cp.cuda.runtime.getDeviceProperties(0)['name'].decode()}")


dnp version : local
numpy       : 2.0.2
torch       : 2.10.0+cu128
CUDA (CuPy) : True  →  device = cuda
CuPy        : 14.0.1
GPU         : Tesla P100-PCIE-16GB


In [ ]:

## 3 · Config
CFG = dict(
    # Audio
    sr          = 32_000,
    duration    = 5,
    n_mels      = 64,
    n_fft       = 1024,
    hop_length  = 320,
    fmin        = 50,
    fmax        = 14_000,

    # Model
    num_classes = 234,

    # Training
    epochs          = 15,
    batch_size      = 32,   # v3 (512-ch) uses ~2× VRAM vs v2; keep 32 for P100
    val_batch_size  = 16,   # smaller batches for safe VRAM headroom in validation
    lr              = 3e-4,
    weight_decay    = 1e-4,
    num_workers     = 4,

    # Device
    device       = DEVICE,

    # Paths
    data_dir     = DATA_DIR,
    out_dir      = OUT_DIR,
    cache_dir    = OUT_DIR + "/mel_cache",   # ~4.3 GB for 35 K clips
)

os.makedirs(CFG["cache_dir"], exist_ok=True)

SAMPLES  = CFG["sr"] * CFG["duration"]
T_FRAMES = 1 + SAMPLES // CFG["hop_length"]
print(f"Mel-spec shape per clip : ({CFG['n_mels']}, {T_FRAMES})")
print(f"Train batch size        : {CFG['batch_size']}")
print(f"Val   batch size        : {CFG['val_batch_size']}")
print(f"Cache dir               : {CFG['cache_dir']}")
print(f"Device                  : {CFG['device']}")


Mel-spec shape per clip : (64, 501)
Batch size              : 64
Device                  : cuda


In [17]:
## 4 · EDA — Load metadata CSVs
train_df    = pd.read_csv(os.path.join(CFG["data_dir"], "train.csv"))
taxonomy_df = pd.read_csv(os.path.join(CFG["data_dir"], "taxonomy.csv"))
labels_df   = pd.read_csv(os.path.join(CFG["data_dir"], "train_soundscapes_labels.csv"))
sample_sub  = pd.read_csv(os.path.join(CFG["data_dir"], "sample_submission.csv"))

# Build label ↔ index mapping from taxonomy  (234 classes, same order as submission)
SPECIES_LIST = taxonomy_df["primary_label"].tolist()
LABEL2IDX    = {lbl: i for i, lbl in enumerate(SPECIES_LIST)}
NUM_CLASSES  = len(SPECIES_LIST)
assert NUM_CLASSES == 234, f"Expected 234 classes, got {NUM_CLASSES}"

print("train.csv        :", train_df.shape)
print("taxonomy.csv     :", taxonomy_df.shape)
print("soundscape labels:", labels_df.shape)
print("sample submission:", sample_sub.shape)
print("\nClasses per taxonomy class:")
print(taxonomy_df["class_name"].value_counts().to_string())
print("\nSpecies list preview:", SPECIES_LIST[:5], "...")

train.csv        : (35549, 15)
taxonomy.csv     : (234, 5)
soundscape labels: (1478, 4)
sample submission: (3, 235)

Classes per taxonomy class:
class_name
Aves        162
Amphibia     35
Insecta      28
Mammalia      8
Reptilia      1

Species list preview: ['1161364', '116570', '1176823', '1491113', '1595929'] ...


In [18]:
## 5 · Audio Utilities

def load_clip(path: str, sr: int = CFG["sr"], duration: int = CFG["duration"]) -> np.ndarray:
    """Load an audio file, pad or crop to *duration* seconds."""
    target_len = sr * duration
    audio, _ = librosa.load(path, sr=sr, mono=True)
    if len(audio) < target_len:
        audio = np.pad(audio, (0, target_len - len(audio)))
    else:
        audio = audio[:target_len]
    return audio.astype(np.float32)


def to_melspec(audio: np.ndarray) -> np.ndarray:
    """Convert raw audio to a normalised log-mel spectrogram (n_mels × T_FRAMES)."""
    mel = librosa.feature.melspectrogram(
        y=audio,
        sr=CFG["sr"],
        n_fft=CFG["n_fft"],
        hop_length=CFG["hop_length"],
        n_mels=CFG["n_mels"],
        fmin=CFG["fmin"],
        fmax=CFG["fmax"],
        power=2.0,
    )
    log_mel = librosa.power_to_db(mel, ref=np.max).astype(np.float32)
    # Normalise to [0, 1]
    log_mel = (log_mel - log_mel.min()) / (log_mel.max() - log_mel.min() + 1e-6)
    return log_mel   # shape: (n_mels, T_FRAMES)


def audio_to_input(path: str) -> np.ndarray:
    """Full pipeline: file → mel → (1, n_mels, T_FRAMES)."""
    audio = load_clip(path)
    mel   = to_melspec(audio)
    return mel[np.newaxis, ...]   # add channel dim


print("Audio utilities ready.")

Audio utilities ready.


In [ ]:

## 6 · PyTorch Datasets + Train/Val Split + Disk-Cache

import hashlib, concurrent.futures
from sklearn.model_selection import train_test_split


def _cache_path(cache_dir: str, filename: str) -> str:
    """Deterministic .npy path — hashed to avoid long filenames."""
    key = hashlib.md5(filename.encode()).hexdigest()
    return os.path.join(cache_dir, key + ".npy")


class BirdTrainDataset(Dataset):
    """
    Each sample: short recording from train_audio/.
    Mel spectrogram is computed once and saved to *cache_dir* as float32 .npy.
    On subsequent accesses (or epochs) the file is loaded directly — no librosa.
    """
    def __init__(self, df: pd.DataFrame, audio_dir: str, label2idx: dict,
                 cache_dir: str = ""):
        self.df        = df.reset_index(drop=True)
        self.audio_dir = audio_dir
        self.label2idx = label2idx
        self.n_classes = len(label2idx)
        self.cache_dir = cache_dir

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row  = self.df.iloc[idx]
        path = os.path.join(self.audio_dir, row["filename"])

        # ── Mel spectrogram (cache-aware) ────────────────────────────────
        if self.cache_dir:
            cp_path = _cache_path(self.cache_dir, row["filename"])
            if os.path.exists(cp_path):
                mel = np.load(cp_path)   # (1, n_mels, T)
            else:
                mel = audio_to_input(path)
                np.save(cp_path, mel)
        else:
            mel = audio_to_input(path)

        # ── Multi-hot label ───────────────────────────────────────────────
        target  = np.zeros(self.n_classes, dtype=np.float32)
        primary = row["primary_label"]
        if primary in self.label2idx:
            target[self.label2idx[primary]] = 1.0
        secondary = row.get("secondary_labels", "[]")
        if isinstance(secondary, str) and secondary not in ("[]", ""):
            for lbl in secondary.strip("[]").replace("'", "").split(","):
                lbl = lbl.strip()
                if lbl in self.label2idx:
                    target[self.label2idx[lbl]] = 1.0

        return torch.from_numpy(mel), torch.from_numpy(target)


class BirdTestDataset(Dataset):
    """5-second windows from test soundscapes."""
    def __init__(self, soundscape_dir: str, sr: int = CFG["sr"],
                 duration: int = CFG["duration"]):
        self.sr       = sr
        self.duration = duration
        self.samples  = []
        for fpath in sorted(Path(soundscape_dir).glob("*.ogg")):
            fname = fpath.stem
            for end_sec in range(duration, 61, duration):
                self.samples.append((str(fpath), fname, end_sec))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        fpath, fname, end_sec = self.samples[idx]
        start_sec = end_sec - self.duration
        audio, _ = librosa.load(fpath, sr=self.sr, mono=True,
                                 offset=start_sec, duration=self.duration)
        if len(audio) < self.sr * self.duration:
            audio = np.pad(audio, (0, self.sr * self.duration - len(audio)))
        audio  = audio.astype(np.float32)
        mel    = to_melspec(audio)[np.newaxis, ...]
        row_id = f"{fname}_{end_sec}"
        return torch.from_numpy(mel), row_id


# ── Filter + stratified 90/10 split ──────────────────────────────────────
valid_mask = train_df["primary_label"].isin(LABEL2IDX)
print(f"Filtered {valid_mask.sum()} / {len(train_df)} recordings in taxonomy.")
clean_df = train_df[valid_mask].copy()

try:
    train_split_df, val_split_df = train_test_split(
        clean_df, test_size=0.1, random_state=42,
        stratify=clean_df["primary_label"],
    )
except ValueError:
    train_split_df, val_split_df = train_test_split(
        clean_df, test_size=0.1, random_state=42,
    )

train_split_df = train_split_df.reset_index(drop=True)
val_split_df   = val_split_df.reset_index(drop=True)

AUDIO_DIR = os.path.join(CFG["data_dir"], "train_audio")

train_dataset = BirdTrainDataset(
    train_split_df, AUDIO_DIR, LABEL2IDX, cache_dir=CFG["cache_dir"])
val_dataset   = BirdTrainDataset(
    val_split_df,   AUDIO_DIR, LABEL2IDX, cache_dir=CFG["cache_dir"])

train_loader = DataLoader(
    train_dataset, batch_size=CFG["batch_size"],  shuffle=True,
    num_workers=CFG["num_workers"], pin_memory=False, drop_last=True,
)
val_loader = DataLoader(
    val_dataset, batch_size=CFG["val_batch_size"], shuffle=False,
    num_workers=CFG["num_workers"], pin_memory=False, drop_last=False,
)
test_dataset = BirdTestDataset(
    soundscape_dir=os.path.join(CFG["data_dir"], "test_soundscapes"),
)
test_loader = DataLoader(
    test_dataset, batch_size=CFG["val_batch_size"], shuffle=False,
    num_workers=CFG["num_workers"],
)

print(f"Train  : {len(train_dataset):>6}  ({len(train_loader)} batches)")
print(f"Val    : {len(val_dataset):>6}  ({len(val_loader)} batches)")
print(f"Test   : {len(test_dataset):>6} windows")

# ── Pre-warm cache (parallel, one-time cost) ─────────────────────────────
def _warm_one(args):
    idx, ds = args
    ds[idx]   # triggers cache write on miss; instant load on hit

all_ds = train_dataset  # val shares same files — train covers all

cache_files = set(os.listdir(CFG["cache_dir"]))
n_total     = len(all_ds)
n_cached    = sum(
    1 for i in range(n_total)
    if os.path.basename(
        _cache_path(CFG["cache_dir"], all_ds.df.iloc[i]["filename"])
    ) in cache_files
)

if n_cached < n_total:
    missing = n_total - n_cached
    print(f"\n── Pre-warming mel cache: {n_cached}/{n_total} already cached, "
          f"computing {missing} ──────────")
    from tqdm.auto import tqdm as _tqdm
    with concurrent.futures.ThreadPoolExecutor(max_workers=CFG["num_workers"]) as ex:
        list(_tqdm(
            ex.map(_warm_one, [(i, all_ds) for i in range(n_total)]),
            total=n_total, unit="clip", desc="  caching",
        ))
    print("  Cache warm-up complete ✅")
else:
    print(f"\n  mel_cache fully populated ({n_total} clips) ✅  — loading from disk.")


Filtered 35549 / 35549 training recordings in taxonomy.
Train  :  31994  (499 batches)
Val    :   3555
Test   :      0 windows


In [ ]:

## 7 · DNP Model — MelSpecCNN-v3 (factored stem · SE-ResNet · MixPool)
#
#  Input : B × 1 × 64 × 501
#  ──────────────────────────────────────────────────────────────────────────
#  Stem     Conv(7×1)+BN+GELU → Conv(1×7)+BN+GELU → MaxPool → B×64×32×250
#  Stage 1  SEResBlock(64)×2  → DownBlock(64→128)           → B×128×16×125
#  Stage 2  SEResBlock(128)×2 → DownBlock(128→256)          → B×256×8×62
#  Stage 3  SEResBlock(256)×2 → DownBlock(256→512)          → B×512×4×31
#  Stage 4  SEResBlock(512)×2 → DownBlock(512→512)          → B×512×2×15
#  Stage 5  SEResBlock(512)                                  → B×512×2×15
#  MixPool  GAP(512) ‖ GMP(512)                             → B×1024
#  Head     Drop(0.4)→FC(1024→512)→GELU→Drop(0.2)→FC(512→234)
#  ≈ 10 M parameters
#  ──────────────────────────────────────────────────────────────────────────

from dnp.core.layers import Conv2d, BatchNorm2d, MaxPool2d, Linear, Dropout
from dnp.core import ops as _ops


class SEBlock(Module):
    """Squeeze-and-Excitation channel attention (Hu et al., 2018)."""
    def __init__(self, channels: int, reduction: int = 8):
        super().__init__()
        mid = max(channels // reduction, 8)
        self.fc1 = Linear(channels, mid, bias=False)
        self.fc2 = Linear(mid, channels, bias=False)

    def forward(self, x: Tensor) -> Tensor:
        B, C = x.shape[0], x.shape[1]
        s = _ops.mean(x, axis=(2, 3))           # (B, C)
        s = _ops.gelu(self.fc1(s))              # (B, C//8)
        s = _ops.sigmoid(self.fc2(s))           # (B, C)
        scale = _ops.reshape(s, (B, C, 1, 1))
        return _ops.multiply(x, scale)


class SEResBlock(Module):
    """SE residual block — preserves channels and spatial dims."""
    def __init__(self, channels: int):
        super().__init__()
        self.conv1 = Conv2d(channels, channels, 3, padding=1)
        self.bn1   = BatchNorm2d(channels)
        self.conv2 = Conv2d(channels, channels, 3, padding=1)
        self.bn2   = BatchNorm2d(channels)
        self.se    = SEBlock(channels)

    def forward(self, x: Tensor) -> Tensor:
        out = _ops.gelu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = self.se(out)
        return _ops.gelu(out + x)

    def set_training(self, flag: bool):
        self.bn1.training = flag
        self.bn2.training = flag

    def get_bn_modules(self):
        return [self.bn1, self.bn2]


class DownBlock(Module):
    """1×1 projection + 2× MaxPool channel expansion / downsampling."""
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        self.conv = Conv2d(in_channels, out_channels, 1)
        self.bn   = BatchNorm2d(out_channels)
        self.pool = MaxPool2d(2, 2)

    def forward(self, x: Tensor) -> Tensor:
        return self.pool(_ops.gelu(self.bn(self.conv(x))))

    def set_training(self, flag: bool):
        self.bn.training = flag

    def get_bn_modules(self):
        return [self.bn]


class MelSpecCNN(Module):
    def __init__(self, num_classes: int = 234):
        super().__init__()
        # Factored stem: frequency axis (7×1) then time axis (1×7)
        # B×1×64×501  →  B×64×64×501  →  B×64×64×501  →  B×64×32×250
        self.stem_conv1 = Conv2d(1,  64, (7, 1), padding=(3, 0))
        self.stem_bn1   = BatchNorm2d(64)
        self.stem_conv2 = Conv2d(64, 64, (1, 7), padding=(0, 3))
        self.stem_bn2   = BatchNorm2d(64)
        self.stem_pool  = MaxPool2d(2, 2)

        # 5 stages of paired SE-ResBlocks + projection DownBlocks
        self.res1a = SEResBlock(64);   self.res1b = SEResBlock(64);   self.down1 = DownBlock(64,  128)
        self.res2a = SEResBlock(128);  self.res2b = SEResBlock(128);  self.down2 = DownBlock(128, 256)
        self.res3a = SEResBlock(256);  self.res3b = SEResBlock(256);  self.down3 = DownBlock(256, 512)
        self.res4a = SEResBlock(512);  self.res4b = SEResBlock(512);  self.down4 = DownBlock(512, 512)
        self.res5  = SEResBlock(512)

        # Classification head (1024 features after MixPool)
        self.drop1 = Dropout(p=0.4)
        self.fc1   = Linear(1024, 512)
        self.drop2 = Dropout(p=0.2)
        self.fc2   = Linear(512, num_classes)

    def forward(self, x: Tensor) -> Tensor:
        # Stem
        x = _ops.gelu(self.stem_bn1(self.stem_conv1(x)))
        x = self.stem_pool(_ops.gelu(self.stem_bn2(self.stem_conv2(x))))
        # Stages  (2 SE-ResBlocks then DownBlock per stage)
        x = self.down1(self.res1b(self.res1a(x)))
        x = self.down2(self.res2b(self.res2a(x)))
        x = self.down3(self.res3b(self.res3a(x)))
        x = self.down4(self.res4b(self.res4a(x)))
        x = self.res5(x)
        # MixPool: GAP ‖ GMP  →  (B, 1024)
        gap = _ops.mean(x, axis=(2, 3))
        gmp = _ops.max(x,  axis=(2, 3))
        x   = _ops.concatenate(gap, gmp, axis=1)
        # Device guard: re-pin features to weights' device.
        # No-op when already on GPU; fixes silent numpy drift on some CuPy builds.
        from dnp.core.backend import get_xp as _gxp
        _xpW = _gxp(self.fc1.W.data)
        if _gxp(x.data) is not _xpW:
            x.data = _xpW.asarray(x.data)
        # Head
        x = self.drop1(x)
        x = _ops.gelu(self.fc1(x))
        x = self.drop2(x)
        return self.fc2(x)

    def _all_bn(self):
        bns = [self.stem_bn1, self.stem_bn2]
        for rb in (self.res1a, self.res1b, self.res2a, self.res2b,
                   self.res3a, self.res3b, self.res4a, self.res4b, self.res5):
            bns += rb.get_bn_modules()
        for db in (self.down1, self.down2, self.down3, self.down4):
            bns += db.get_bn_modules()
        return bns

    def train_mode(self):
        self.training = True
        self.drop1.training = self.drop2.training = True
        for bn in self._all_bn(): bn.training = True
        for rb in (self.res1a, self.res1b, self.res2a, self.res2b,
                   self.res3a, self.res3b, self.res4a, self.res4b, self.res5):
            rb.set_training(True)
        for db in (self.down1, self.down2, self.down3, self.down4):
            db.set_training(True)

    def eval_mode(self):
        self.training = False
        self.drop1.training = self.drop2.training = False
        for bn in self._all_bn(): bn.training = False
        for rb in (self.res1a, self.res1b, self.res2a, self.res2b,
                   self.res3a, self.res3b, self.res4a, self.res4b, self.res5):
            rb.set_training(False)
        for db in (self.down1, self.down2, self.down3, self.down4):
            db.set_training(False)

    def cuda(self):
        """Move all parameters and BatchNorm running stats to GPU (CuPy)."""
        from dnp.core.backend import as_cupy
        for p in self.parameters():
            p.data = as_cupy(p.data)
            if p.grad is not None:
                p.grad = as_cupy(p.grad)
        for bn in self._all_bn():
            bn.running_mean = as_cupy(bn.running_mean)
            bn.running_var  = as_cupy(bn.running_var)
        return self


# ── Instantiate → GPU first → optimizer ──────────────────────────────────
model = MelSpecCNN(num_classes=NUM_CLASSES)

if CFG["device"] == "cuda":
    model.cuda()
    print("Model moved to GPU ✅")

from dnp.core.optimizers import AdamW
criterion = BCEWithLogitsLoss()
optimizer = AdamW(model.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"])
scheduler = ReduceLROnPlateau(optimizer, patience=3, factor=0.5)

total_params = sum(p.data.size for p in model.parameters())
print(f"Model parameters : {total_params:,}  (~{total_params/1e6:.1f} M)")

# Quick shape sanity-check
import numpy as _np_check
_dummy   = Tensor(_np_check.zeros((2, 1, 64, 501), dtype=_np_check.float32),
                  device=CFG["device"])
with session.no_grad():
    _out = model(_dummy)
session.reset()
print(f"Forward check    : input (2,1,64,501) → output {_out.shape}  ✅")
del _dummy, _out, _np_check
print("Model & optimizer ready.")


Model moved to GPU ✅
Model parameters : 1,865,962  (~1.9 M)
Forward check    : input (2,1,64,501) → output (2, 234)  ✅
Model & optimizer ready.


In [ ]:

## 8 · Training Loop  (tqdm · validation · AUC / F1)

import time, gc
import numpy as np
from tqdm.auto import tqdm
from sklearn.metrics import roc_auc_score, f1_score

USE_CUDA     = CFG["device"] == "cuda"
history      = {"train_loss": [], "val_loss": [], "val_auc": [], "val_f1": []}
best_val_auc = 0.0


def _to_device(arr: np.ndarray):
    return as_cupy(arr) if USE_CUDA else arr


def _free_vram():
    """Defragment CuPy memory pool + run Python GC."""
    if USE_CUDA:
        import cupy as cp
        cp.get_default_memory_pool().free_all_blocks()
        cp.get_default_pinned_memory_pool().free_all_blocks()
    gc.collect()


def _sigmoid(x: np.ndarray) -> np.ndarray:
    return 1.0 / (1.0 + np.exp(-np.clip(x, -88.0, 88.0)))


def validate_epoch(mdl, loader):
    """No-grad validation pass → (val_loss, macro-AUC, macro-F1)."""
    mdl.eval_mode()
    all_logits, all_labels = [], []
    total_loss, n_batches  = 0.0, 0

    for mel_batch, label_batch in loader:
        x_arr = _to_device(mel_batch.numpy().astype(np.float32))
        y_arr = _to_device(label_batch.numpy().astype(np.float32))

        with session.no_grad():
            logits = mdl.forward(Tensor(x_arr, name="xv", device=CFG["device"]))
            loss   = criterion(
                logits,
                Tensor(y_arr, name="yv", device=CFG["device"]),
            )

        total_loss += float(as_numpy(loss.data))
        all_logits.append(as_numpy(logits.data))
        all_labels.append(label_batch.numpy().astype(np.float32))
        n_batches  += 1
        session.reset()           # free graph after each val batch
        del x_arr, y_arr, logits, loss

    mdl.train_mode()

    logits_np = np.vstack(all_logits)   # raw logits (no sigmoid)
    labels_np = np.vstack(all_labels)
    probs     = _sigmoid(logits_np)
    preds     = (probs >= 0.5).astype(np.float32)

    # AUC: use raw logits, restrict to classes that have at least one positive
    mask = labels_np.sum(axis=0) > 0
    try:
        auc = roc_auc_score(labels_np[:, mask], logits_np[:, mask], average="macro")
    except Exception:
        auc = float("nan")

    f1 = f1_score(labels_np, preds, average="macro", zero_division=0)
    return total_loss / max(n_batches, 1), auc, f1


# ── Main training loop ────────────────────────────────────────────────────
session.reset()

epoch_bar = tqdm(range(1, CFG["epochs"] + 1), desc="Epochs", unit="ep",
                 dynamic_ncols=True)

for epoch in epoch_bar:
    model.train_mode()
    epoch_loss = 0.0
    t0 = time.time()

    batch_bar = tqdm(
        train_loader,
        desc=f"  Train {epoch:02d}/{CFG['epochs']}",
        leave=False, unit="batch", dynamic_ncols=True,
    )
    for step, (mel_batch, label_batch) in enumerate(batch_bar):
        x_arr = _to_device(mel_batch.numpy().astype(np.float32))
        y_arr = _to_device(label_batch.numpy().astype(np.float32))

        with session.graph():
            logits = model(Tensor(x_arr, name="x", device=CFG["device"]))
            loss   = criterion(logits, Tensor(y_arr, name="y", device=CFG["device"]))

        loss.backward()
        optimizer.step()
        optimizer.zero_grad()   # zero grads BEFORE session.reset()
        session.reset()

        if USE_CUDA:
            synchronize()

        epoch_loss += float(as_numpy(loss.data))
        del x_arr, y_arr, logits, loss
        batch_bar.set_postfix(loss=f"{epoch_loss / (step + 1):.4f}")

    avg_train_loss = epoch_loss / len(train_loader)

    # ── Free VRAM before validation so we start with a clean memory pool ─
    _free_vram()

    val_loss, val_auc, val_f1 = validate_epoch(model, val_loader)

    history["train_loss"].append(avg_train_loss)
    history["val_loss"].append(val_loss)
    history["val_auc"].append(val_auc)
    history["val_f1"].append(val_f1)

    scheduler.step(val_loss)
    _free_vram()

    # Checkpoint best model by val AUC (the competition metric)
    if val_auc > best_val_auc:
        best_val_auc = val_auc
        np.savez(
            os.path.join(CFG["out_dir"], "best_model.npz"),
            **{f"p{i}": as_numpy(p.data) for i, p in enumerate(model.parameters())},
        )

    epoch_bar.set_postfix(
        tr=f"{avg_train_loss:.4f}",
        vl=f"{val_loss:.4f}",
        auc=f"{val_auc:.3f}",
        f1=f"{val_f1:.3f}",
        best=f"{best_val_auc:.3f}",
        t=f"{time.time()-t0:.0f}s",
    )

print(f"\nTraining complete.  Best val AUC: {best_val_auc:.4f}")


Epochs:   0%|          | 0/15 [00:00<?, ?ep/s]

  Train 01/15:   0%|          | 0/499 [00:00<?, ?batch/s]

OutOfMemoryError: Out of memory allocating 589,824,000 bytes (allocated so far: 16,385,239,040 bytes).

In [ ]:

## 9 · Inference & Submission

def sigmoid_np(x: np.ndarray) -> np.ndarray:
    return 1.0 / (1.0 + np.exp(-np.clip(x, -88.0, 88.0)))


model.eval_mode()

all_row_ids = []
all_preds   = []

for mel_batch, row_ids in test_loader:
    x_arr = _to_device(mel_batch.numpy().astype(np.float32))

    with session.no_grad():
        x_t    = Tensor(x_arr, name="x_test", device=CFG["device"])
        logits = model.forward(x_t)

    if USE_CUDA:
        synchronize()

    probs = sigmoid_np(as_numpy(logits.data).astype(np.float32))
    all_row_ids.extend(list(row_ids))
    all_preds.append(probs)
    session.reset()

all_preds = np.vstack(all_preds)

sub_df = pd.DataFrame(all_preds, columns=SPECIES_LIST)
sub_df.insert(0, "row_id", all_row_ids)
sub_df = sub_df.set_index("row_id").reindex(sample_sub["row_id"]).reset_index()
sub_df = sub_df.fillna(0.0)

out_path = os.path.join(CFG["out_dir"], "submission.csv")
sub_df.to_csv(out_path, index=False)

print(f"Submission saved → {out_path}")
print(f"Shape: {sub_df.shape}")
sub_df.head(3)


In [ ]:

## 10 · Learning Curves
import matplotlib.pyplot as plt

epochs_ran = range(1, len(history["train_loss"]) + 1)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle("BirdCLEF+ 2026 — Training Curves", fontsize=13, fontweight="bold")

axes[0].plot(epochs_ran, history["train_loss"], "o-", label="Train")
axes[0].plot(epochs_ran, history["val_loss"],   "s--", label="Val")
axes[0].set_title("BCE Loss"); axes[0].set_xlabel("Epoch")
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(epochs_ran, history["val_auc"], "^-", color="green")
axes[1].set_title("Val macro-AUC (↑ competition metric)")
axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("AUC")
axes[1].axhline(max(history["val_auc"]), color="gray", ls=":", lw=1)
axes[1].grid(alpha=0.3)

axes[2].plot(epochs_ran, history["val_f1"], "D-", color="purple")
axes[2].set_title("Val macro-F1 (threshold 0.5)")
axes[2].set_xlabel("Epoch"); axes[2].set_ylabel("F1")
axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(CFG["out_dir"], "training_curves.png"), dpi=120)
plt.show()

best_ep = history["val_auc"].index(max(history["val_auc"])) + 1
print(f"Best val AUC : {max(history['val_auc']):.4f}  (epoch {best_ep})")
print(f"Best val F1  : {max(history['val_f1']):.4f}")
